# model_inference - საბოლოო Kaggle submission

ეს notebook ტვირთავს პროექტის საუკეთესო მოდელს (**`XGBoost_BEST`**) MLflow **Model Registry**-დან და
აგენერირებს საბოლოო submission-ს raw `test.csv`-ზე.


In [2]:
import importlib.util, os, subprocess, sys
from pathlib import Path

IN_COLAB = importlib.util.find_spec("google.colab") is not None
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    ROOT = Path("/content/drive/MyDrive/ML_FINAL_PROJECT")
else:
    ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents]
                if (p / "data" / "train.csv.zip").exists())

for pkg in ["xgboost", "mlflow"]:
    if importlib.util.find_spec(pkg) is None:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], check=True)

sys.path.insert(0, str(ROOT))
print("ROOT =", ROOT)

Mounted at /content/drive
ROOT = /content/drive/MyDrive/ML_FINAL_PROJECT


In [3]:
import numpy as np
import pandas as pd
import mlflow

from src.experiment_utils import setup_mlflow

test = pd.read_csv(ROOT / "data/test.csv.zip")

DAGSHUB_REPO = "ZukaCS/ML_FINAL_PROJECT"
print("tracking:", setup_mlflow(ROOT, dagshub_repo=DAGSHUB_REPO))
print("test", test.shape)

/usr/local/lib/python3.12/dist-packages/mlflow/pyfunc/utils/data_validation.py:187: UserWarning: Add type hints to the `predict` method to enable data validation and automatic signature inference during model logging. Check https://mlflow.org/docs/latest/model/python_model.html#type-hint-usage-in-pythonmodel for more details.
  color_warning(


DagsHub username (leave blank to log locally): bed4c06fe3c1f8a3cb658c93c03a14c39c3bb0d3
DagsHub token: ··········
tracking: https://dagshub.com/ZukaCS/ML_FINAL_PROJECT.mlflow
test (115064, 4)


In [4]:
from mlflow.tracking import MlflowClient

MODEL_NAME = "XGBoost_BEST"
client = MlflowClient()

try:
    versions = client.search_model_versions(f"name='{MODEL_NAME}'")
except Exception:
    versions = [v for v in client.search_model_versions() if v.name == MODEL_NAME]
latest = max(versions, key=lambda v: int(v.version))
MODEL_URI = f"models:/{MODEL_NAME}/{latest.version}"

print(f"loading {MODEL_URI}  (run_id = {latest.run_id})")
model = mlflow.pyfunc.load_model(MODEL_URI)
print("model loaded")

loading models:/XGBoost_BEST/1  (run_id = d57bc74779574d839e915eef7ef779c4)


/usr/local/lib/python3.12/dist-packages/mlflow/pyfunc/model.py:840: UserWarning: [17:42:39] WARNING: /__w/xgboost/xgboost/src/gbm/gbtree.cc:443: Changing updater from `grow_gpu_hist` to `grow_quantile_histmaker`.
  return cloudpickle.load(f)
/usr/local/lib/python3.12/dist-packages/mlflow/pyfunc/model.py:840: UserWarning: [17:42:39] WARNING: /__w/xgboost/xgboost/src/context.cc:55: No visible GPU is found, setting device to CPU.
  return cloudpickle.load(f)
/usr/local/lib/python3.12/dist-packages/mlflow/pyfunc/model.py:840: UserWarning: [17:42:39] WARNING: /__w/xgboost/xgboost/src/context.cc:210: Device is changed from GPU to CPU as we couldn't find any available GPU on the system.
  return cloudpickle.load(f)


model loaded


In [5]:

preds = np.asarray(model.predict(test), dtype=float)
assert len(preds) == len(test) and not np.isnan(preds).any()

submission = pd.DataFrame({
    "Id": test["Store"].astype(str) + "_" + test["Dept"].astype(str)
          + "_" + test["Date"].astype(str),
    "Weekly_Sales": preds,
})

sub_dir = ROOT / "submissions"
sub_dir.mkdir(exist_ok=True)
sub_path = sub_dir / "submission_final.csv"
submission.to_csv(sub_path, index=False)
print("wrote", sub_path, "| rows:", len(submission))
submission.head()

wrote /content/drive/MyDrive/ML_FINAL_PROJECT/submissions/submission_final.csv | rows: 115064


,Id,Weekly_Sales
0,1_1_2012-11-02,37489.675781
1,1_1_2012-11-09,20658.773438
2,1_1_2012-11-16,19847.142578
3,1_1_2012-11-23,20566.583984
4,1_1_2012-11-30,26087.800781
